# Кластеризация записей дневника

Анализ тематических кластеров на основе векторных эмбеддингов записей пользователей.

**Методы:**
- Векторизация текста: `fastembed` BAAI/bge-small-en-v1.5 (384 dims)
- Кластеризация: K-Means
- Подбор числа кластеров: Silhouette Score
- Снижение размерности для визуализации: PCA (384 → 2D)

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from dotenv import load_dotenv

load_dotenv('../.env')
print('OK')

In [ ]:
import asyncio
import asyncpg

async def load_embeddings(user_id: int):
    conn = await asyncpg.connect(os.getenv('DATABASE_URL').replace('+asyncpg', ''))
    rows = await conn.fetch(
        "SELECT id, text, created_at, embedding::text FROM entries "
        "WHERE user_id=$1 AND embedding IS NOT NULL AND text IS NOT NULL "
        "ORDER BY created_at",
        user_id
    )
    await conn.close()
    return rows

USER_ID = int(input('user_id: '))
rows = asyncio.run(load_embeddings(USER_ID))
print(f'Загружено {len(rows)} записей')

In [ ]:
import ast

texts = [r['text'] for r in rows]
dates = [r['created_at'] for r in rows]
X = np.array([ast.literal_eval(r['embedding']) for r in rows], dtype=np.float32)

print(f'Матрица эмбеддингов: {X.shape}  (записей × размерность)')

## 1. Подбор оптимального числа кластеров (Silhouette Score)

In [ ]:
max_k = min(10, len(rows) - 1)
scores = []
ks = range(2, max_k + 1)

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(X)
    scores.append(silhouette_score(X, labels))

best_k = ks[np.argmax(scores)]

plt.figure(figsize=(8, 4))
plt.plot(list(ks), scores, marker='o', color='steelblue')
plt.axvline(best_k, color='tomato', linestyle='--', label=f'Оптимум k={best_k}')
plt.xlabel('Число кластеров k')
plt.ylabel('Silhouette Score')
plt.title('Подбор числа кластеров')
plt.legend()
plt.tight_layout()
plt.savefig('silhouette_scores.png', dpi=150)
plt.show()
print(f'Лучший k={best_k}, score={max(scores):.3f}')

## 2. Silhouette-диаграмма для лучшего k

In [ ]:
km = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
labels = km.fit_predict(X)
sample_scores = silhouette_samples(X, labels)

fig, ax = plt.subplots(figsize=(8, 5))
y_lower = 10
colors = cm.tab10(np.linspace(0, 1, best_k))

for i in range(best_k):
    cluster_scores = np.sort(sample_scores[labels == i])
    size = len(cluster_scores)
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_scores, color=colors[i], alpha=0.7)
    ax.text(-0.05, y_lower + size / 2, str(i + 1))
    y_lower = y_upper + 10

ax.axvline(silhouette_score(X, labels), color='red', linestyle='--', label='Среднее')
ax.set_xlabel('Silhouette Score')
ax.set_ylabel('Кластер')
ax.set_title(f'Silhouette-диаграмма (k={best_k})')
ax.legend()
plt.tight_layout()
plt.savefig('silhouette_diagram.png', dpi=150)
plt.show()

## 3. Визуализация кластеров (PCA 384 → 2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)
explained = pca.explained_variance_ratio_.sum()

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab10', alpha=0.7, s=40)
plt.colorbar(scatter, label='Кластер')
plt.title(f'Кластеры записей дневника (PCA, объяснённая дисперсия: {explained:.1%})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.savefig('clusters_pca.png', dpi=150)
plt.show()
print(f'PCA объясняет {explained:.1%} дисперсии')

## 4. Примеры записей по кластерам

In [ ]:
for cluster_id in range(best_k):
    idxs = [i for i, l in enumerate(labels) if l == cluster_id]
    print(f'\n--- Тема {cluster_id + 1} ({len(idxs)} записей) ---')
    for i in idxs[:3]:
        preview = texts[i][:120].replace('\n', ' ') if texts[i] else ''
        print(f'  {dates[i].strftime("%d.%m.%Y")}: {preview}…')